# 04 - Feature Engineering

# Imports and Load Data

In [27]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df = pd.read_csv('../data/processed/cleaned_dataset.csv')
print('shape:', df.shape)

shape: (88167, 20)


# Step 1 - Drop Weak and Redundant Features

In [28]:
df.drop(columns=['key', 'time_signature', 'loudness'], inplace=True)
print('shape after dropping weak features:', df.shape)
print('remaining columns:', df.columns.tolist())

shape after dropping weak features: (88167, 17)
remaining columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'track_genre']


# Step 2 - Convert Duration to Minutes

In [29]:
df['duration_min'] = (df['duration_ms'] / 60000).round(2)
df.drop(columns=['duration_ms'], inplace=True)
print('shape:', df.shape)
print('duration_min sample:', df['duration_min'].head().tolist())

shape: (88167, 17)
duration_min sample: [3.84, 2.49, 3.51, 3.37, 3.31]


# Step 3 - Create Mood Categories

In [30]:
def get_mood(row, threshold=0.5):
    if row['valence'] >= threshold and row['energy'] >= threshold:
        return 'happy'
    elif row['valence'] < threshold and row['energy'] >= threshold:
        return 'angry'
    elif row['valence'] >= threshold and row['energy'] < threshold:
        return 'calm'
    else:
        return 'sad'

df['mood'] = df.apply(lambda row: get_mood(row, threshold=0.5), axis=1)
df['mood_v2'] = df.apply(lambda row: get_mood(row, threshold=0.4), axis=1)

print('mood v1 distribution (threshold 0.5):')
print(df['mood'].value_counts())
print('\nmood v2 distribution (threshold 0.4):')
print(df['mood_v2'].value_counts())

mood v1 distribution (threshold 0.5):
mood
happy    32992
angry    29754
sad      18252
calm      7169
Name: count, dtype: int64

mood v2 distribution (threshold 0.4):
mood_v2
happy    44808
angry    26282
sad      11528
calm      5549
Name: count, dtype: int64


# Step 4 - Encode Mood Categories

In [31]:
mood_map = {'happy': 0, 'angry': 1, 'sad': 2, 'calm': 3}

df['mood_encoded'] = df['mood'].map(mood_map)
df['mood_v2_encoded'] = df['mood_v2'].map(mood_map)

print('mood encoded sample:')
print(df[['mood', 'mood_encoded', 'mood_v2', 'mood_v2_encoded']].head())

mood encoded sample:
   mood  mood_encoded mood_v2  mood_v2_encoded
0  calm             3   happy                0
1   sad             2     sad                2
2   sad             2     sad                2
3   sad             2     sad                2
4   sad             2   angry                1


# Step 5 - Create Tempo Categories

In [32]:
def get_tempo_category(tempo):
    if tempo < 90:
        return 'slow'
    elif tempo <= 140:
        return 'medium'
    else:
        return 'fast'

df['tempo_category'] = df['tempo'].apply(get_tempo_category)
print('tempo category distribution:')
print(df['tempo_category'].value_counts())

tempo category distribution:
tempo_category
medium    52435
fast      22928
slow      12804
Name: count, dtype: int64


# Step 6 - Encode Tempo Categories

In [33]:
tempo_map = {'slow': 0, 'medium': 1, 'fast': 2}
df['tempo_encoded'] = df['tempo_category'].map(tempo_map)

print('tempo encoded sample:')
print(df[['tempo_category', 'tempo_encoded']].value_counts().sort_index())

tempo encoded sample:
tempo_category  tempo_encoded
fast            2                22928
medium          1                52435
slow            0                12804
Name: count, dtype: int64


# Step 7 - Create Popularity Tiers

In [34]:
def get_popularity_tier(popularity):
    if popularity <= 25:
        return 'emerging'
    elif popularity <= 50:
        return 'upcoming'
    elif popularity <= 75:
        return 'mainstream'
    else:
        return 'charttoppers'

df['popularity_tier'] = df['popularity'].apply(get_popularity_tier)
print('popularity tier distribution:')
print(df['popularity_tier'].value_counts())

popularity tier distribution:
popularity_tier
upcoming        34318
emerging        34093
mainstream      18479
charttoppers     1277
Name: count, dtype: int64


# Step 8 - Encode Popularity Tiers

In [35]:
popularity_map = {'emerging': 0, 'upcoming': 1, 'mainstream': 2, 'charttoppers': 3}
df['popularity_tier_encoded'] = df['popularity_tier'].map(popularity_map)

print('popularity tier encoded sample:')
print(df[['popularity_tier', 'popularity_tier_encoded']].value_counts().sort_index())

popularity tier encoded sample:
popularity_tier  popularity_tier_encoded
charttoppers     3                           1277
emerging         0                          34093
mainstream       2                          18479
upcoming         1                          34318
Name: count, dtype: int64


# Step 9 - Create Combined Features

In [36]:
df['dance_energy_score'] = df['danceability'] * df['energy']
df['positive_energy'] = df['valence'] * df['energy']
df['vocal_score'] = 1 - df['instrumentalness']
df['energy_acousticness_ratio'] = np.log1p(df['energy']) - np.log1p(df['acousticness'])

print('combined features sample:')
print(df[['dance_energy_score', 'positive_energy', 
          'vocal_score', 'energy_acousticness_ratio']].describe().round(3))

combined features sample:
       dance_energy_score  positive_energy  vocal_score  \
count           88167.000        88167.000    88167.000   
mean                0.366            0.318        0.831   
std                 0.179            0.225        0.321   
min                 0.000            0.000        0.000   
25%                 0.234            0.126        0.917   
50%                 0.379            0.284        1.000   
75%                 0.499            0.481        1.000   
max                 0.956            0.972        1.000   

       energy_acousticness_ratio  
count                  88167.000  
mean                       0.229  
std                        0.381  
min                       -0.691  
25%                       -0.049  
50%                        0.336  
75%                        0.558  
max                        0.693  


# Step 10 - Create Artist Count Feature

In [37]:
df['artist_count'] = df['artists'].str.split(';').str.len()
print('artist count distribution:')
print(df['artist_count'].value_counts().sort_index().head(10))

artist count distribution:
artist_count
1     66001
2     15589
3      4555
4      1228
5       399
6       159
7        95
8        48
9        20
10       22
Name: count, dtype: int64


# Step 11 - Create Genre Count Feature

In [38]:
df['genre_count'] = df['track_genre'].str.split(', ').str.len()
print('genre count distribution:')
print(df['genre_count'].value_counts().sort_index())

genre count distribution:
genre_count
1    72007
2    11306
3     2940
4     1357
5      430
6      103
7       21
8        2
9        1
Name: count, dtype: int64


# Step 12 - Create TF-IDF Text Feature

In [39]:
df['tfidf_input'] = (
    df['track_genre'].str.replace(', ', ' ') + ' ' +
    df['mood'] + ' ' +
    df['tempo_category'] + ' ' +
    df['popularity_tier']
)

print('tfidf_input sample:')
print(df['tfidf_input'].head(5).tolist())

tfidf_input sample:
['acoustic j-pop singer-songwriter songwriter calm slow mainstream', 'acoustic chill sad slow mainstream', 'acoustic sad slow mainstream', 'acoustic sad fast mainstream', 'acoustic sad medium charttoppers']


# Step 13 - Define Feature Sets

In [40]:
audio_features = [
    'danceability', 'energy', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_min', 'explicit'
]

full_features = [
    'danceability', 'energy', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness',
    'valence', 'tempo', 'duration_min', 'explicit',
    'dance_energy_score', 'positive_energy', 'vocal_score',
    'energy_acousticness_ratio', 'artist_count', 'genre_count',
    'mood_encoded', 'mood_v2_encoded', 'tempo_encoded',
    'popularity_tier_encoded'
]

print('audio features:', len(audio_features))
print('full features:', len(full_features))

audio features: 11
full features: 21


# Step 14 - Scale Feature Sets

In [41]:
# full features
scaler_standard_full = StandardScaler()
scaler_minmax_full = MinMaxScaler()

df_full_standard = df.copy()
df_full_minmax = df.copy()
df_full_standard[full_features] = scaler_standard_full.fit_transform(df[full_features])
df_full_minmax[full_features] = scaler_minmax_full.fit_transform(df[full_features])

# audio features
scaler_standard_audio = StandardScaler()
scaler_minmax_audio = MinMaxScaler()

df_audio_standard = df.copy()
df_audio_minmax = df.copy()
df_audio_standard[audio_features] = scaler_standard_audio.fit_transform(df[audio_features])
df_audio_minmax[audio_features] = scaler_minmax_audio.fit_transform(df[audio_features])

print('full features standard scaled:', df_full_standard[full_features].shape)
print('full features minmax scaled:', df_full_minmax[full_features].shape)
print('audio features standard scaled:', df_audio_standard[audio_features].shape)
print('audio features minmax scaled:', df_audio_minmax[audio_features].shape)

full features standard scaled: (88167, 21)
full features minmax scaled: (88167, 21)
audio features standard scaled: (88167, 11)
audio features minmax scaled: (88167, 11)


# Step 15 - Save All Datasets

In [42]:
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/featured_dataset.csv', index=False)
df_full_standard.to_csv('../data/processed/featured_full_standard.csv', index=False)
df_full_minmax.to_csv('../data/processed/featured_full_minmax.csv', index=False)
df_audio_standard.to_csv('../data/processed/featured_audio_standard.csv', index=False)
df_audio_minmax.to_csv('../data/processed/featured_audio_minmax.csv', index=False)

print('saved files:')
print('1. featured_dataset.csv          → unscaled all features')
print('2. featured_full_standard.csv    → full features standardized')
print('3. featured_full_minmax.csv      → full features normalized')
print('4. featured_audio_standard.csv   → audio features standardized')
print('5. featured_audio_minmax.csv     → audio features normalized')

saved files:
1. featured_dataset.csv          → unscaled all features
2. featured_full_standard.csv    → full features standardized
3. featured_full_minmax.csv      → full features normalized
4. featured_audio_standard.csv   → audio features standardized
5. featured_audio_minmax.csv     → audio features normalized


# Step 16 - Save Feature Set Definitions

In [43]:
feature_sets = {
    'audio_features': audio_features,
    'full_features': full_features
}

with open('../config/feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=4)

print('saved feature sets to config/feature_sets.json')
print('\naudio_features:', len(audio_features), 'features')
print('full_features:', len(full_features), 'features')

saved feature sets to config/feature_sets.json

audio_features: 11 features
full_features: 21 features


# Feature Engineering Summary

In [44]:
print('='*50)
print('FEATURE ENGINEERING SUMMARY')
print('='*50)

print('\n--- Dropped Features ---')
print('key            → weak effect on popularity')
print('time_signature → weak effect on popularity')
print('loudness       → correlated with energy (0.76)')
print('duration_ms    → converted to duration_min')

print('\n--- New Features Created ---')
print('duration_min              → duration in minutes')
print('mood                      → happy/angry/sad/calm (threshold 0.5)')
print('mood_v2                   → happy/angry/sad/calm (threshold 0.4)')
print('mood_encoded              → mood as numbers')
print('mood_v2_encoded           → mood_v2 as numbers')
print('tempo_category            → slow/medium/fast')
print('tempo_encoded             → slow=0, medium=1, fast=2')
print('popularity_tier           → emerging/upcoming/mainstream/charttoppers')
print('popularity_tier_encoded   → emerging=0, upcoming=1, mainstream=2, charttoppers=3')
print('dance_energy_score        → danceability * energy')
print('positive_energy           → valence * energy')
print('vocal_score               → 1 - instrumentalness')
print('energy_acousticness_ratio → log(energy) - log(acousticness)')
print('artist_count              → number of artists per song')
print('genre_count               → number of genres per song')
print('tfidf_input               → text for TF-IDF model')

print('\n--- Feature Sets ---')
print(f'audio_features → {len(audio_features)} features')
print(f'full_features  → {len(full_features)} features')

print('\n--- Scaling ---')
print('StandardScaler → standardization (mean=0, std=1)')
print('MinMaxScaler   → normalization (min=0, max=1)')
print('applied to both audio and full feature sets')

print('\n--- Saved Files ---')
print('featured_dataset.csv         → unscaled all features')
print('featured_full_standard.csv   → full features standardized')
print('featured_full_minmax.csv     → full features normalized')
print('featured_audio_standard.csv  → audio features standardized')
print('featured_audio_minmax.csv    → audio features normalized')
print('config/feature_sets.json     → feature set definitions')

print('\n--- Final Shape ---')
print(f'rows: {df.shape[0]}')
print(f'total columns: {df.shape[1]}')

FEATURE ENGINEERING SUMMARY

--- Dropped Features ---
key            → weak effect on popularity
time_signature → weak effect on popularity
loudness       → correlated with energy (0.76)
duration_ms    → converted to duration_min

--- New Features Created ---
duration_min              → duration in minutes
mood                      → happy/angry/sad/calm (threshold 0.5)
mood_v2                   → happy/angry/sad/calm (threshold 0.4)
mood_encoded              → mood as numbers
mood_v2_encoded           → mood_v2 as numbers
tempo_category            → slow/medium/fast
tempo_encoded             → slow=0, medium=1, fast=2
popularity_tier           → emerging/upcoming/mainstream/charttoppers
popularity_tier_encoded   → emerging=0, upcoming=1, mainstream=2, charttoppers=3
dance_energy_score        → danceability * energy
positive_energy           → valence * energy
vocal_score               → 1 - instrumentalness
energy_acousticness_ratio → log(energy) - log(acousticness)
artist_count      